In [9]:
pip install -U evaluate

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import pandas as pd
import numpy as np
import torch
import evaluate
import torch.nn as nn
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import pipeline , DataCollatorWithPadding  , AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

C:\Users\saini\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
data = pd.read_csv('D:/Machine Learning/DeepLearing Projects/news_dataset.csv')  
data.head()

,title,text,subject,date,label
0,WATCH: Trump Just Told All The Anti-Gay Bigot...,A whole lot of evangelical Trump voters just d...,News,"November 13, 2016",0
1,"China backs U.N. call for justice in Yemen, U....",GENEVA (Reuters) - China signaled on Wednesday...,worldnews,"September 13, 2017",1
2,THE PEOPLE’S PRESIDENT: Trump Meets With Coal ...,,politics,"Feb 16, 2017",0
3,WSJ REPORTER RIPS INTO DEM CANDIDATES For Thei...,THE WSJ S MARY KISSEL NAILS IT ON THE DEM DEBA...,politics,"Nov 15, 2015",0
4,Lawmakers aim to delay U.S. ceding control of ...,WASHINGTON (Reuters) - Critics of a plan for t...,politicsNews,"September 13, 2016",1


In [12]:
data.shape

(6000, 5)

In [13]:
data.isnull().sum()  # null values in the dataset

title      0
text       0
subject    0
date       0
label      0
dtype: int64

In [14]:
# Full text of the text and title
data['full_text'] = data['title'] + ' ' + data['text']

In [15]:
data.head()

,title,text,subject,date,label,full_text
0,WATCH: Trump Just Told All The Anti-Gay Bigot...,A whole lot of evangelical Trump voters just d...,News,"November 13, 2016",0,WATCH: Trump Just Told All The Anti-Gay Bigot...
1,"China backs U.N. call for justice in Yemen, U....",GENEVA (Reuters) - China signaled on Wednesday...,worldnews,"September 13, 2017",1,"China backs U.N. call for justice in Yemen, U...."
2,THE PEOPLE’S PRESIDENT: Trump Meets With Coal ...,,politics,"Feb 16, 2017",0,THE PEOPLE’S PRESIDENT: Trump Meets With Coal ...
3,WSJ REPORTER RIPS INTO DEM CANDIDATES For Thei...,THE WSJ S MARY KISSEL NAILS IT ON THE DEM DEBA...,politics,"Nov 15, 2015",0,WSJ REPORTER RIPS INTO DEM CANDIDATES For Thei...
4,Lawmakers aim to delay U.S. ceding control of ...,WASHINGTON (Reuters) - Critics of a plan for t...,politicsNews,"September 13, 2016",1,Lawmakers aim to delay U.S. ceding control of ...


In [16]:
data.drop(columns=  ['title', 'text' , "subject" , "date"], axis=1, inplace=True)

In [17]:
data

,label,full_text
0,0,WATCH: Trump Just Told All The Anti-Gay Bigot...
1,1,"China backs U.N. call for justice in Yemen, U...."
2,0,THE PEOPLE’S PRESIDENT: Trump Meets With Coal ...
3,0,WSJ REPORTER RIPS INTO DEM CANDIDATES For Thei...
4,1,Lawmakers aim to delay U.S. ceding control of ...
...,...,...
5995,1,Exiled Venezuelan opposition magistrates resur...
5996,1,Plague outbreak in Madagascar kills 20: WHO NA...
5997,1,"Trump to visit Asia in November, North Korea i..."
5998,1,Melania Trump calls taped comments by Donald T...


In [18]:
tarin , test = train_test_split(data  , random_state=11)
tarin.shape , test.shape

((4500, 2), (1500, 2))

In [19]:
x = data.drop(columns=['label' , 'full_text'])
y = data['label']

In [20]:
x_train , x_test , y_train , y_test = train_test_split(x , y , random_state=11)
x_train.shape , x_test.shape , y_train.shape , y_test.shape

((4500, 0), (1500, 0), (4500,), (1500,))

In [21]:
train_dataset = Dataset.from_pandas(tarin , preserve_index=False)
test_dataset = Dataset.from_pandas(test , preserve_index=False)

In [22]:
print(train_dataset)
print(test_dataset)

Dataset({
    features: ['label', 'full_text'],
    num_rows: 4500
})
Dataset({
    features: ['label', 'full_text'],
    num_rows: 1500
})


In [23]:
model_name = 'distilbert/distilbert-base-uncased-finetuned-sst-2-english'

In [24]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name , num_labels=2,
    ignore_mismatched_sizes=True)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 3565.33it/s]


In [25]:
def preprocessing(dataset):


    
    def tokenize_data(item):
        return tokenizer(item["full_text"] , truncation=True)
    encoded_data =  dataset.map(tokenize_data , batched=True)

    encoded_data.set_format("torch" , columns=["input_ids", "attention_mask", "label"])
    return encoded_data

preprocessing(train_dataset)

Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

Map: 100%|██████████| 4500/4500 [00:02<00:00, 1943.06 examples/s]


Dataset({
    features: ['label', 'full_text', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 4500
})

In [26]:
train_encoded = preprocessing(train_dataset)
test_encoded = preprocessing(test_dataset)

Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

Map: 100%|██████████| 1500/1500 [00:00<00:00, 1530.35 examples/s]


In [27]:
metric = evaluate.combine(["accuracy", "f1", "precision", "recall"])

In [28]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)
training_args = TrainingArguments(
    output_dir="./fake_news_model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,         
    per_device_eval_batch_size=4,          
    gradient_accumulation_steps=4,       
    fp16=True,                           
    num_train_epochs=3,
    weight_decay=0.01,
    save_strategy="epoch"
)

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset=train_encoded,
    eval_dataset = test_encoded,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)
trainer.train()

C:\Users\saini\AppData\Roaming\Python\Python314\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


In [ ]:
model.save_pretrained("./fake_news_model")
tokenizer.save_pretrained("./fake_news_model")

In [ ]:
classifier = pipeline("text-classification", model="./fake_news_model", tokenizer="./fake_news_model")